# 05d · Product similarity (Node Similarity), corrected
Replaces section 6 of `05b`. There the projection was built as `User -> Product`, so GDS compared **users**, not products (the `SIMILAR` relationships it wrote joined users and the product example table came back empty). Here products are the *source* nodes, so Jaccard similarity is computed between products over the sets of users who carted or bought them.

Projection: cart and purchase interactions, products with at least 20 such users. Top 5 neighbours per product, similarity >= 0.01 (a first run with 0.1 left only 158 pairs: Jaccard between buyer sets is naturally low, max 0.22). Metrics: `ml/outputs/05d_metrics.json`.

In [1]:
import os, json, time
import pandas as pd
from IPython.display import display
from neo4j import GraphDatabase
driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]))
OUT = "/home/jovyan/work/ml/outputs"; metrics = {}
def q(query, **p):
    with driver.session(database="neo4j") as s: return s.run(query, **p).data()
def save(): json.dump(metrics, open(f"{OUT}/05d_metrics.json", "w"), indent=2, default=str)

## 1. Remove the mistaken user–user `SIMILAR` relationships from `05b`

In [2]:
before = q("MATCH ()-[s:SIMILAR|SIMILAR_TO]->() RETURN count(s) AS n")[0]["n"]
q("MATCH ()-[s:SIMILAR|SIMILAR_TO]->() CALL (s) { DELETE s } IN TRANSACTIONS OF 200000 ROWS")
metrics["removed_previous_similarity_rels"] = before; print("removed", before)

removed 158


## 2. Projection with products as sources

In [3]:
q("CALL gds.graph.drop('buy', false)")
t0 = time.time()
g = q("""MATCH (u:User)-[r:INTERACTED]->(p:Product) WHERE r.type IN ['cart','purchase']
    WITH p, u, sum(r.count) AS c WITH p, collect([u, c]) AS pairs WHERE size(pairs) >= 20
    UNWIND pairs AS pr WITH p, pr[0] AS u, pr[1] AS c
    RETURN gds.graph.project('buy', p, u, {sourceNodeLabels: labels(p), targetNodeLabels: labels(u), relationshipProperties:{count:c}}) AS g""")[0]["g"]
metrics["projection"] = dict(nodeCount=g["nodeCount"], relationshipCount=g["relationshipCount"], seconds=round(time.time()-t0, 1))
print(metrics["projection"]); save()

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('buy', false)"


{'nodeCount': 1116927, 'relationshipCount': 2168965, 'seconds': 46.2}


## 3. Node similarity

In [4]:
t0 = time.time()
ns = q("""CALL gds.nodeSimilarity.write('buy', {writeRelationshipType:'SIMILAR_TO', writeProperty:'jaccard',
    topK:5, similarityCutoff:0.01, concurrency:4})
    YIELD nodesCompared, relationshipsWritten, similarityDistribution RETURN nodesCompared, relationshipsWritten,
    similarityDistribution.mean AS meanSim, similarityDistribution.p50 AS p50, similarityDistribution.max AS maxSim""")[0]
ns["seconds"] = round(time.time()-t0, 1); metrics["node_similarity"] = ns; print(ns); save()

{'nodesCompared': 11890, 'relationshipsWritten': 58233, 'meanSim': 0.025478853657451454, 'p50': 0.021739184856414795, 'maxSim': 0.21739196777343747, 'seconds': 8.6}


In [5]:
labels = q("MATCH (a:Product)-[s:SIMILAR_TO]->(b:Product) RETURN count(DISTINCT a) AS products_with_neighbours, count(s) AS pairs")[0]
metrics["coverage"] = labels
ex = pd.DataFrame(q("""MATCH (a:Product)-[s:SIMILAR_TO]->(b:Product) WITH a, b, s ORDER BY a.pagerank DESC, s.jaccard DESC LIMIT 12
    RETURN a.productId AS a_id, a.brand AS brand_a, a.categoryCode AS cat_a, b.productId AS b_id, b.brand AS brand_b, b.categoryCode AS cat_b, round(s.jaccard,3) AS jaccard"""))
same_cat = q("""MATCH (a:Product)-[s:SIMILAR_TO]->(b:Product) WHERE a.categoryCode IS NOT NULL AND b.categoryCode IS NOT NULL
    RETURN avg(CASE WHEN a.categoryCode = b.categoryCode THEN 1.0 ELSE 0.0 END) AS same_category,
           avg(CASE WHEN split(a.categoryCode,'.')[0] = split(b.categoryCode,'.')[0] THEN 1.0 ELSE 0.0 END) AS same_top_level,
           avg(CASE WHEN a.brand = b.brand THEN 1.0 ELSE 0.0 END) AS same_brand""")[0]
metrics["similarity_examples"] = ex.to_dict(orient="records"); metrics["agreement"] = same_cat
display(ex); print(same_cat); save()

,a_id,brand_a,cat_a,b_id,brand_b,cat_b,jaccard
0,1004856,samsung,electronics.smartphone,1004833,samsung,electronics.smartphone,0.076
1,1004856,samsung,electronics.smartphone,1004767,samsung,electronics.smartphone,0.060
2,1004856,samsung,electronics.smartphone,1005100,samsung,electronics.smartphone,0.057
3,1004856,samsung,electronics.smartphone,1004858,samsung,electronics.smartphone,0.043
4,1004856,samsung,electronics.smartphone,1004857,samsung,electronics.smartphone,0.031
5,1004767,samsung,electronics.smartphone,1004870,samsung,electronics.smartphone,0.085
6,1004767,samsung,electronics.smartphone,1004856,samsung,electronics.smartphone,0.060
7,1004767,samsung,electronics.smartphone,1004833,samsung,electronics.smartphone,0.057
8,1004767,samsung,electronics.smartphone,1004768,samsung,electronics.smartphone,0.052
9,1004767,samsung,electronics.smartphone,1004836,samsung,electronics.smartphone,0.050


{'same_category': 0.7695511858300812, 'same_top_level': 0.8734989492644838, 'same_brand': 0.4448739117382164}


In [6]:
q("CALL gds.graph.drop('buy', false)"); save(); driver.close()

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('buy', false)"
